In [ ]:
# Celda 1: Entorno y carga
import runpy
from pathlib import Path
from pyspark.sql import functions as F

runpy.run_path(str(next(p / "notebooks" / "_init_env.py" for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "notebooks" / "_init_env.py").is_file())))

from src.catalog import (
    EXTREMADURA_LAT_MAX,
    EXTREMADURA_LAT_MIN,
    EXTREMADURA_LON_MAX,
    EXTREMADURA_LON_MIN,
    GOLD_FIRMS_GRID_TABLE,
    GOLD_FIRMS_MONTHLY_TABLE,
    GOLD_FIRMS_PREV_7D_TABLE,
    SILVER_FIRMS_TABLE,
)

df_silver = spark.table(SILVER_FIRMS_TABLE)
df_grid = spark.table(GOLD_FIRMS_GRID_TABLE)
df_monthly = spark.table(GOLD_FIRMS_MONTHLY_TABLE)
df_prev7d = spark.table(GOLD_FIRMS_PREV_7D_TABLE)

In [ ]:
# Celda 2: Checks de calidad espacial y de FRP
assert df_silver.filter(
    ~F.col("latitude").between(EXTREMADURA_LAT_MIN, EXTREMADURA_LAT_MAX)
    | ~F.col("longitude").between(EXTREMADURA_LON_MIN, EXTREMADURA_LON_MAX)
).count() == 0, "Coordenadas fuera de Extremadura"
assert df_silver.filter(F.col("frp") < 0).count() == 0, "FRP negativo"
assert all(t.count() > 0 for t in [df_silver, df_grid, df_monthly, df_prev7d]), "Alguna tabla FIRMS está vacía"
print("Checks FIRMS OK. Detecciones Silver:", df_silver.count())

In [ ]:
# Celda 3: Celdas con mayor energía radiativa acumulada (FRP)
display(df_grid.orderBy(F.desc("frp_sum")).limit(15))

In [ ]:
# Celda 4: Meses pico y actividad previa de 7 días en la misma celda
display(df_monthly.orderBy(F.desc("frp_sum")).limit(12))
display(df_prev7d.filter(F.col("prev_7d_detections") > 0).orderBy(F.desc("prev_7d_frp_sum")).limit(15))